In [2]:
import pandas as pd
from astropy.table import Table
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from collections import defaultdict
import os

In [ ]:
len(Table.read('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/FP_SGAFPVI.fits'))

In [ ]:
# Load y3 FP Sample
fp_targets = pd.read_csv("/global/cfs/cdirs/desi/science/td/pv/fpgalaxies/Y3/v1/loa_FP_targets_v1.csv")
# Preview the first few rows
print(fp_targets.head())

# Load John Lucy's visually inspected sample
fpvi = pd.read_csv('/global/cfs/cdirs/desi/science/td/pv/fpgalaxies/Y1/v2/jrl_spirals_FP_pv_cat_v2.csv', delim_whitespace=True)
print(fpvi.head())

## Investigating Duplicates in y3 Sample

In [ ]:
total = len(fp_targets)
unique = fp_targets["targetid"].nunique()
print(f"Total Rows: {total}")
print(f"Unique targetid values: {unique}")
print(f"Number of extra rows: {total - unique}")

duplicates = fp_targets[fp_targets.duplicated(subset='targetid', keep=False)]

In [ ]:
# Count how many times each targetid appears
dup_counts = fp_targets['targetid'].value_counts()

# Filter only those with more than one appearance
dup_counts = dup_counts[dup_counts > 1]
print(dup_counts.head())
print(f"Number of galaxies duplicated: {len(dup_counts)}")

# Count how many targetids have 2, 3, 4, ... occurrences
dup_distribution = dup_counts.value_counts().sort_index()
print(dup_distribution)

In [ ]:
# Get only the first appearance of each duplicated targetid
first_appearances = fp_targets[fp_targets['targetid'].isin(dup_counts.index)]
first_appearances = first_appearances.drop_duplicates(subset='targetid', keep='first')
first_appearances

In [ ]:
# Drop duplicate targetids, keeping the first occurrence
fp_targets_unique = fp_targets.drop_duplicates(subset='targetid', keep='first')

# Confirm result
print(f"Original number of rows: {len(fp_targets)}")
print(f"After dropping duplicates: {len(fp_targets_unique)}")
print(f"Number of unique targetids: {fp_targets_unique['targetid'].nunique()}")

In [ ]:
fp_targets_unique.to_csv("fp_targets_unqiue.csv")

In [ ]:
# Ensure both are the same type (int64 is safe)
fpy3 = set(fp_targets_unique["targetid"].astype(int))
fpy1vi = set(fpvi["targetID"].astype(int))

# Compute overlaps and differences
common_ids = fpy3 & fpy1vi
only_in_y3 = fpy3 - fpy1vi
only_in_y1 = fpy1vi - fpy3

# Print summary
print(f"Total Y3 targets: {len(fpy3)}")
print(f"Total VIY1 targets: {len(fpy1vi)}")
print(f"Common targets: {len(common_ids)}")
print(f"Targets only in Y3: {len(only_in_y3)}")
print(f"Targets only in VIY1: {len(only_in_y1)}")

## Which FP targets are close to any SGA Galaxies?

In [ ]:
# Implementing a Cone Search
from astropy.coordinates import SkyCoord
import astropy.units as u

sga2020 = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', hdu=1)

# Create SkyCoord objects
fp_coords = SkyCoord(ra=fp_targets_unique["target_ra"].values * u.deg,
                     dec=fp_targets_unique["target_dec"].values * u.deg)

sga_coords = SkyCoord(ra=sga2020["RA"] * u.deg,
                      dec=sga2020["DEC"] * u.deg)

# Do a cone search (cross-match) within 10 arcseconds
idx_fp, idx_sga, sep2d, _ = fp_coords.search_around_sky(sga_coords, 10 * u.arcsec)

# This gives you the indices of matches from both catalogs
print(f"Number of matched targets: {len(idx_fp)}")

In [ ]:
# Load morphology classifications
morph_df = pd.read_csv('/global/cfs/cdirs/desi/science/td/pv/SGA2020/classifications_SGA2020_and_vi.csv')
morph_df

In [ ]:
# Create DataFrame from matched indices
matched_df = pd.DataFrame({
    "SGA_ID": sga2020["SGA_ID"][idx_fp],
    "targetid": fp_targets_unique["targetid"].values[idx_sga],
    "Separation_arcsec": sep2d.arcsec
})
print(matched_df.head(11))

In [ ]:
# Merge with morphology classification on SGA_ID
merged_df = matched_df.merge(morph_df[["SGA_ID", "Classification"]], on="SGA_ID", how="left")

In [ ]:
print(merged_df.dtypes)

In [ ]:
# Convert SGA_ID to native-endian int64
merged_df["SGA_ID"] = merged_df["SGA_ID"].values.byteswap().newbyteorder().astype(np.int64)

# Sort by targetid and Separation_arcsec
merged_df_sorted = merged_df.sort_values(by=["targetid", "Separation_arcsec"])

# Keep only the closest match per targetid
merged_df_dedup = merged_df_sorted.drop_duplicates(subset="targetid", keep="first")

# Re-run your classification filter
classified_only_df = merged_df_dedup[merged_df_dedup["Classification"] != "Unclassified"]
unclassified_df = merged_df_dedup[merged_df_dedup["Classification"] == "Unclassified"]

# Confirm result
print(f"Deduplicated classified_only_df size: {len(classified_only_df)}")
print(f"Unique targetids: {classified_only_df['targetid'].nunique()}")

In [ ]:
print(merged_df_dedup.head(10))
print(classified_only_df['Classification'].value_counts(dropna=False))

In [ ]:
#classified_only_df.to_csv('/global/cfs/cdirs/desi/science/td/pv/fpgalaxies/Y3/v1/FP_SGA2020_Morphs.csv')

In [ ]:
classified_ids = classified_only_df["targetid"].unique()
fp_targets_cleaned = fp_targets_unique[~fp_targets_unique["targetid"].isin(classified_ids)]
print(f"Original fp_targets_unique: {len(fp_targets_unique)}")
print(f"After removing classified galaxies: {len(fp_targets_cleaned)}")

In [ ]:
#fp_targets_cleaned.to_csv('FP_Targets_SSL.csv')

## Looking at Anchor Sample within FP_Targets_SSL

In [3]:
fp_ssl_sample = pd.read_csv('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/FP_Targets_SSL.csv')
print(f"fp sample to classify with ssl: {len(fp_ssl_sample)}")

fp_vi_sample = pd.read_csv('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/Generating_Anchor_Points/fp_vi_all_morphologies.csv')
print(f"fp vi sample: {len(fp_vi_sample)}")

fp_unique_sample = pd.read_csv('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/fp_targets_unique.csv')
print(f"fp unique sample: {len(fp_unique_sample)}")

fp sample to classify with ssl: 160444
fp vi sample: 1600
fp unique sample: 230216


In [4]:
fp_vi_sample

,fp_targetid,fp_ra,fp_dec,dr8_id,gz_ra,gz_dec,has-spiral-arms_yes_fraction,smooth-or-featured_smooth_fraction,smooth-or-featured_featured-or-disk_fraction,smooth-or-featured_artifact_fraction,merging_major-disturbance_fraction,merging_minor-disturbance_fraction,merging_merger_fraction,sep_arcsec,morphology
0,39627984172222271,38.716912,8.177774,377919_532,38.716909,8.177771,NaN,0.69,0.10,0.21,0.04,0.07,0.08,0.014074,elliptical
1,39632970251767515,133.639531,34.513842,518117_2742,133.639531,34.513842,NaN,0.78,0.13,0.08,0.02,0.05,0.01,0.001770,elliptical
2,39628343213036459,7.593253,23.827344,463521_4208,7.593253,23.827344,NaN,0.82,0.06,0.13,0.02,0.05,0.01,0.002158,elliptical
3,39633236116112574,164.866659,49.136781,581504_139,164.866659,49.136781,NaN,0.75,0.18,0.07,0.02,0.09,0.01,0.001536,elliptical
4,39633025352335769,217.286644,37.138578,531254_330,217.286644,37.138578,NaN,0.83,0.09,0.08,0.03,0.09,0.01,0.000874,elliptical
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1595,39627767142163066,31.990039,-0.699797,326175_4827,31.990039,-0.699797,NaN,0.62,0.28,0.10,0.04,0.10,0.19,0.001174,irregular
1596,2842507742281729,128.472746,-0.715745,326561_3524,128.472746,-0.715745,NaN,0.63,0.24,0.13,0.15,0.11,0.39,0.001477,irregular
1597,39628102359318661,1.287069,13.329428,406097_3479,1.287067,13.329424,NaN,0.64,0.20,0.16,0.05,0.11,0.34,0.014639,irregular
1598,39633445114088154,97.232180,65.326482,631333_3055,97.232180,65.326482,NaN,0.45,0.45,0.10,0.08,0.09,0.64,0.000733,irregular


In [5]:
fp_unique_sample

,Unnamed: 0,targetid,survey,program,healpix,brickid,brickname,brick_objid,release,target_ra,...,in_south,filename,zcmb,ppxf_vdisp,ppxf_vdisp_err,ppxf_chi2,ppxf_corrected_err,ppxf_snr,dnr_snr,ppxf_dof
0,0,2398569927540753,main,bright,4188,389714,1433p102,4709,9010,143.419416,...,True,loa/healpix/main/bright/41/4188/redrock-main-b...,0.010967,162.952087,137.167167,0.322612,77.909466,0.645730,1.163921,2560.0
1,1,2411549142548491,main,dark,8193,338472,2261p012,968,9010,226.025162,...,True,loa/healpix/main/dark/81/8193/redrock-main-dar...,0.092164,178.287028,10.689031,0.653099,8.638291,3.874081,1.629334,2257.0
2,2,2411764067074053,main,dark,4188,389714,1433p102,4709,9010,143.419530,...,True,loa/healpix/main/dark/41/4188/redrock-main-dar...,0.011021,9.460587,27.703287,0.514055,19.862594,1.898028,1.587233,2560.0
3,3,2411812238655501,main,dark,27765,401199,1875p122,5568,9010,187.662823,...,True,loa/healpix/main/dark/277/27765/redrock-main-d...,0.086665,93.994346,12.905137,0.637026,10.300098,3.795211,1.666452,2291.0
4,4,2411853036650517,main,dark,27604,410926,1618p140,4583,9010,161.913662,...,True,loa/healpix/main/dark/276/27604/redrock-main-d...,0.094612,95.068810,10.926552,0.692103,9.090099,4.781902,1.660396,2247.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
230211,234556,2305843047855821074,main,backup,36853,317597,0473m022,1699,9012,47.326368,...,True,loa/healpix/main/backup/368/36853/redrock-main...,0.041891,98.281155,8.706692,0.731166,7.444940,5.236558,1.663610,2521.0
230212,234557,2305843047860016411,main,backup,36858,316140,0431m025,336,9010,43.006785,...,True,loa/healpix/main/backup/368/36858/redrock-main...,0.082714,147.400078,18.649748,0.518109,13.424048,3.507441,1.621793,2300.0
230213,234558,2305843056437369156,main,backup,45038,311094,2209m035,3467,9010,220.938119,...,True,loa/healpix/main/backup/450/45038/redrock-main...,0.044581,173.123453,18.635270,0.582172,14.218754,4.217440,1.634469,2515.0
230214,234559,2305843056441572262,main,backup,45043,311110,2249m035,5463,9010,225.021203,...,True,loa/healpix/main/backup/450/45043/redrock-main...,0.091159,229.651775,24.056161,0.495015,16.925272,3.229393,1.615585,2262.0


In [6]:
fp_ssl_sample

,targetid,survey,program,healpix,brickid,brickname,brick_objid,release,target_ra,target_dec,...,zcmb,ppxf_vdisp,ppxf_vdisp_err,ppxf_chi2,ppxf_corrected_err,ppxf_snr,dnr_snr,ppxf_dof,Train-Test_Split,fullpath
0,2398569927540753,main,bright,4188,389714,1433p102,4709,9010,143.419416,10.132312,...,0.010967,162.952087,137.167167,0.322612,77.909466,0.645730,1.163921,2560.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...
1,2411764067074053,main,dark,4188,389714,1433p102,4709,9010,143.419530,10.132452,...,0.011021,9.460587,27.703287,0.514055,19.862594,1.898028,1.587233,2560.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...
2,2411812238655501,main,dark,27765,401199,1875p122,5568,9010,187.662823,12.372729,...,0.086665,93.994346,12.905137,0.637026,10.300098,3.795211,1.666452,2291.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...
3,2411853036650517,main,dark,27604,410926,1618p140,4583,9010,161.913662,13.966366,...,0.094612,95.068810,10.926552,0.692103,9.090099,4.781902,1.660396,2247.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...
4,2802785447837696,main,dark,23052,293212,0564m065,3346,9010,56.428994,-6.497341,...,0.091417,159.040205,4.958651,0.859621,4.597451,4.876959,1.626049,2257.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160439,2305843047855821074,main,backup,36853,317597,0473m022,1699,9012,47.326368,-2.371212,...,0.041891,98.281155,8.706692,0.731166,7.444940,5.236558,1.663610,2521.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...
160440,2305843047860016411,main,backup,36858,316140,0431m025,336,9010,43.006785,-2.516770,...,0.082714,147.400078,18.649748,0.518109,13.424048,3.507441,1.621793,2300.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...
160441,2305843056437369156,main,backup,45038,311094,2209m035,3467,9010,220.938119,-3.496219,...,0.044581,173.123453,18.635270,0.582172,14.218754,4.217440,1.634469,2515.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...
160442,2305843056441572262,main,backup,45043,311110,2249m035,5463,9010,225.021203,-3.471301,...,0.091159,229.651775,24.056161,0.495015,16.925272,3.229393,1.615585,2262.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...


In [10]:
# normalize whitespace just in case
fp_ssl_sample["targetid"] = fp_ssl_sample["targetid"].str.strip()
fp_vi_sample["fp_targetid"]  = fp_vi_sample["fp_targetid"].str.strip()

matching = fp_vi_sample[fp_vi_sample["fp_targetid"].isin(fp_ssl_sample["targetid"])]
print(len(matching))

1600


In [14]:
fp_ellip = pd.read_csv('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/Generating_Anchor_Points/fp_vi_elliptical_correct.csv')
print(f"fp ellipticals: {len(fp_ellip)}")

fp_len = pd.read_csv('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/Generating_Anchor_Points/fp_vi_lenticular_correct.csv')
print(f"fp lenticulars: {len(fp_len)}")

fp_spiral = pd.read_csv('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/Generating_Anchor_Points/fp_vi_spiral_correct.csv')
print(f"fp spirals: {len(fp_spiral)}")

fp_irreg = pd.read_csv('/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/Generating_Anchor_Points/fp_vi_irregular_correct.csv')
print(f"fp irregulars: {len(fp_irreg)}")

fp ellipticals: 400
fp lenticulars: 400
fp spirals: 400
fp irregulars: 400


In [24]:
print("fp sample to classify with ssl:", len(fp_ssl_sample))
print("fp vi sample:", len(fp_vi_sample))

# Keep only necessary columns from VI sample and remove duplicates
vi_map = (
    fp_vi_sample[["fp_targetid", "morphology"]]
    .dropna(subset=["fp_targetid"])
    .drop_duplicates(subset=["fp_targetid"], keep="first")
)

# LEFT merge so we keep all SSL galaxies
merged = fp_ssl_sample.merge(
    vi_map,
    left_on="targetid",
    right_on="fp_targetid",
    how="left"
)

# Drop the extra key column from VI table
merged = merged.drop(columns=["fp_targetid"])

# Fill missing morphologies
merged["morphology"] = merged["morphology"].fillna("Unknown")

# Add Train-Test Split column
merged["Train-Test_Split"] = np.where(
    merged["morphology"] != "Unknown",
    "train",
    "test"
)

print("\nTrain/Test counts:")
print(merged["Train-Test_Split"].value_counts())

print("\nMorphology counts:")
print(merged["morphology"].value_counts())

fp sample to classify with ssl: 160444
fp vi sample: 1600

Train/Test counts:
Train-Test_Split
test     158844
train      1600
Name: count, dtype: int64

Morphology counts:
morphology
Unknown       158844
irregular        400
lenticular       400
spiral           400
elliptical       400
Name: count, dtype: int64


In [25]:
merged

,targetid,survey,program,healpix,brickid,brickname,brick_objid,release,target_ra,target_dec,...,ppxf_vdisp,ppxf_vdisp_err,ppxf_chi2,ppxf_corrected_err,ppxf_snr,dnr_snr,ppxf_dof,Train-Test_Split,fullpath,morphology
0,2398569927540753,main,bright,4188,389714,1433p102,4709,9010,143.419416,10.132312,...,162.952087,137.167167,0.322612,77.909466,0.645730,1.163921,2560.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown
1,2411764067074053,main,dark,4188,389714,1433p102,4709,9010,143.419530,10.132452,...,9.460587,27.703287,0.514055,19.862594,1.898028,1.587233,2560.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown
2,2411812238655501,main,dark,27765,401199,1875p122,5568,9010,187.662823,12.372729,...,93.994346,12.905137,0.637026,10.300098,3.795211,1.666452,2291.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown
3,2411853036650517,main,dark,27604,410926,1618p140,4583,9010,161.913662,13.966366,...,95.068810,10.926552,0.692103,9.090099,4.781902,1.660396,2247.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown
4,2802785447837696,main,dark,23052,293212,0564m065,3346,9010,56.428994,-6.497341,...,159.040205,4.958651,0.859621,4.597451,4.876959,1.626049,2257.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160439,2305843047855821074,main,backup,36853,317597,0473m022,1699,9012,47.326368,-2.371212,...,98.281155,8.706692,0.731166,7.444940,5.236558,1.663610,2521.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown
160440,2305843047860016411,main,backup,36858,316140,0431m025,336,9010,43.006785,-2.516770,...,147.400078,18.649748,0.518109,13.424048,3.507441,1.621793,2300.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown
160441,2305843056437369156,main,backup,45038,311094,2209m035,3467,9010,220.938119,-3.496219,...,173.123453,18.635270,0.582172,14.218754,4.217440,1.634469,2515.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown
160442,2305843056441572262,main,backup,45043,311110,2249m035,5463,9010,225.021203,-3.471301,...,229.651775,24.056161,0.495015,16.925272,3.229393,1.615585,2262.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown


In [26]:
# Map morphology -> numeric Main_Type code
morph_to_main_type = {
    "irregular":  -5,
    "lenticular":  0,
    "spiral":     10,
    "elliptical": 20,
    "Unknown":    30
}

# If your morphology strings might have capitalization, normalize first
merged["morphology"] = merged["morphology"].astype(str).str.strip()

# Create Main_Type (numeric)
merged["Main_Type"] = merged["morphology"].map(morph_to_main_type).fillna(30).astype(int)

In [27]:
print(merged[["morphology", "Main_Type"]].value_counts().sort_index())

morphology  Main_Type
Unknown      30          158844
elliptical   20             400
irregular   -5              400
lenticular   0              400
spiral       10             400
Name: count, dtype: int64


In [28]:
merged

,targetid,survey,program,healpix,brickid,brickname,brick_objid,release,target_ra,target_dec,...,ppxf_vdisp_err,ppxf_chi2,ppxf_corrected_err,ppxf_snr,dnr_snr,ppxf_dof,Train-Test_Split,fullpath,morphology,Main_Type
0,2398569927540753,main,bright,4188,389714,1433p102,4709,9010,143.419416,10.132312,...,137.167167,0.322612,77.909466,0.645730,1.163921,2560.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown,30
1,2411764067074053,main,dark,4188,389714,1433p102,4709,9010,143.419530,10.132452,...,27.703287,0.514055,19.862594,1.898028,1.587233,2560.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown,30
2,2411812238655501,main,dark,27765,401199,1875p122,5568,9010,187.662823,12.372729,...,12.905137,0.637026,10.300098,3.795211,1.666452,2291.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown,30
3,2411853036650517,main,dark,27604,410926,1618p140,4583,9010,161.913662,13.966366,...,10.926552,0.692103,9.090099,4.781902,1.660396,2247.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown,30
4,2802785447837696,main,dark,23052,293212,0564m065,3346,9010,56.428994,-6.497341,...,4.958651,0.859621,4.597451,4.876959,1.626049,2257.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown,30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160439,2305843047855821074,main,backup,36853,317597,0473m022,1699,9012,47.326368,-2.371212,...,8.706692,0.731166,7.444940,5.236558,1.663610,2521.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown,30
160440,2305843047860016411,main,backup,36858,316140,0431m025,336,9010,43.006785,-2.516770,...,18.649748,0.518109,13.424048,3.507441,1.621793,2300.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown,30
160441,2305843056437369156,main,backup,45038,311094,2209m035,3467,9010,220.938119,-3.496219,...,18.635270,0.582172,14.218754,4.217440,1.634469,2515.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown,30
160442,2305843056441572262,main,backup,45043,311110,2249m035,5463,9010,225.021203,-3.471301,...,24.056161,0.495015,16.925272,3.229393,1.615585,2262.0,test,/global/cfs/cdirs/desi/spectro/redux/loa/healp...,Unknown,30


In [30]:
out_path = '/pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/FP_Targets_SSL_wanchors.csv'

merged.to_csv(out_path, index=False)

print("Saved to:", out_path)

Saved to: /pscratch/sd/j/jlargett/DESI_SGA_MINE/Sorter/FP/FP_Targets_SSL_wanchors.csv
